# Featuresmith Tutorial: Rule Engine Configs & Findings

In this tutorial, we will use the **Titanic** dataset to explore how Featuresmith evaluates quality contracts, flags missing metrics, and isolates crashed rules.

---

In [ ]:
import os

import featuresmith as fs

dataset_path = os.path.join("..", "data", "processed", "titanic.csv")
dataset = fs.load(dataset_path)

### Step 1: Default Analysis Run

Let's execute all rules enabled by default.

In [ ]:
result = fs.analyze(dataset)

print(f"Triggered findings count: {len(result.findings)}")
for finding in result.findings[:5]:
    print(
        f"- [{finding.severity.upper()}] Column: {finding.column_name} | Rule: {finding.rule_id}"
    )
    print(f"  {finding.title}: {finding.description}")

### Step 2: Customizing Rule Thresholds

Featuresmith lets you inject custom parameters at call-time. Let's configure `quality.missing_value_threshold` to trigger warnings when missingness is above 10% (instead of default 20%).

In [ ]:
custom_config = {"quality.missing_value_threshold": {"threshold": 10.0}}

result_custom = fs.analyze(dataset, rule_config=custom_config)

print(f"Triggered findings count with custom threshold: {len(result_custom.findings)}")

### Step 3: Gating Specific Rules

If you only want to evaluate a subset of rules, pass their IDs to `enabled_rules`.

In [ ]:
result_gated = fs.analyze(
    dataset, enabled_rules=["quality.fully_empty_columns", "quality.duplicate_rows"]
)

print(f"Triggered findings count: {len(result_gated.findings)}")
print(f"Rules evaluated: {result_gated.executed_rules}")

### Step 4: Error Isolation

If a rule throws an exception (due to bad code or invalid states), Featuresmith catches the error internally and registers it in `result.failed_rules` rather than aborting the pipeline. Let's check if any rules failed.

In [ ]:
print(f"Failed rules count: {len(result.failed_rules)}")